# 선택 활동 5 항공사의 고객 만족도를 높이기 위한 고객 서비스 모델 만들기

교과서 116~117쪽

항공사는 고객의 만족도를 높이기 위해 어떤 서비스에 중점을 두어야 할까? 만족도 조사 데이터로 분류 모델을 만들고, 결정트리의 루트 노드와 특징 중요도로 가장 중요한 서비스를 찾아보자.

**데이터**: 교과서 116쪽 링크(캐글)의 `airline_passenger_satisfaction.csv`

In [ ]:
#@title 데이터 준비하기
how = "사이트에서 바로 받기" #@param ["사이트에서 바로 받기", "내 컴퓨터에서 올리기"]
#@markdown 디지털 교과서에 올려 둔 교과서 자료(airline_passenger_satisfaction.csv)를 코랩으로 바로 가져옵니다. 직접 모은 파일을 쓰려면 '내 컴퓨터에서 올리기'를 고르세요.
if how.startswith('사이트'):
    import urllib.request
    for name in ['airline_passenger_satisfaction.csv']:
        urllib.request.urlretrieve('https://aibook-dl.vercel.app/data/' + name, name)
        print('받았습니다:', name)
else:
    from google.colab import files
    uploaded = files.upload()


In [ ]:
#@title ① 데이터 불러오고 특징·타깃 정하기
import pandas as pd
df = pd.read_csv('airline_passenger_satisfaction.csv')
X = df.iloc[:, 9:23]            # 특징: 9~22번 열 (서비스 만족도 14개 항목, 0~5점)
y = df.iloc[:, -1]              # 타깃: 마지막 열 (Satisfaction)
print(X.shape, y.shape)
print('특징:', list(X.columns))
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [ ]:
#@title ② 결정트리와 로지스틱 회귀로 학습하고 평가하기
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
dt = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print(f'결정트리 정확도: {dt.score(X_test, y_test):.3f}')
print(f'로지스틱 회귀 정확도: {lr.score(X_test, y_test):.3f}')

In [ ]:
#@title ③ 결정트리 그리기와 특징 중요도 (교과서 117쪽)
import numpy as np
import matplotlib.pyplot as plt
plt.figure(figsize=(20, 10))
plot_tree(dt, feature_names=X_train.columns, max_depth=2, filled=True)
plt.show()
imp = pd.Series(dt.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(figsize=(8, 6)); plt.title('Feature importance'); plt.show()
print('가장 중요한 서비스:', imp.index[-1], round(imp.iloc[-1], 2))

### 생각해 보기
- 루트 노드의 특징과 특징 중요도 1위는 같은가요? 항공사는 어떤 서비스에 먼저 투자해야 할까요?
- 만족도 항목들은 모두 0~5점으로 범위가 같아서 따로 정규화하지 않았습니다. 범위가 다른 특징이 섞여 있다면 어떻게 해야 할까요?